In [26]:
import pandas as pd
import pickle
import requests

from io import StringIO
from string import ascii_uppercase as alphabet

In [3]:
all_tables = pd.read_html('https://en.wikipedia.org/wiki/2022_FIFA_World_Cup')

HTTPError: HTTP Error 403: Forbidden

In [4]:
url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

print(response.status_code)

200


In [6]:
all_tables = pd.read_html(StringIO(response.text))

In [20]:
all_tables[9]
all_tables[11]
all_tables[31]

,Pos,Teamvte,Pld,W,D,L,GF,GA,GD,Pts,Qualification
0,1,England,3,2,1,0,6,2,+4,7,Advance to knockout stage
1,2,Croatia,3,2,0,1,5,5,0,6,Advance to knockout stage
2,3,Ghana,3,1,1,1,2,2,0,4,Advance to knockout stage
3,4,Panama,3,0,0,3,0,4,−4,0,NaN


In [29]:
dict_table = {}
for letter, i in zip(alphabet, range(9, 33, 2)):
    df = all_tables[i]
    df.rename(columns={df.columns[1]:'Team'}, inplace=True)
    df.pop('Qualification')
    dict_table[f'Group {letter}'] = df

In [31]:
dict_table['Group A']

,Pos,Team,Pld,W,D,L,GF,GA,GD,Pts
0,1,Mexico (H),3,3,0,0,6,0,+6,9
1,2,South Africa,3,1,1,1,2,3,−1,4
2,3,South Korea,3,1,0,2,2,3,−1,3
3,4,Czech Republic,3,0,1,2,2,6,−4,1


In [32]:
with open('dict_table', 'wb') as output:
    pickle.dump(dict_table, output)

In [36]:
import os

os.listdir()

['.ipynb_checkpoints', '1_get_tables_groupstage_2026.ipynb', 'dict_table']

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

years = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]



def get_matches(year):
    web = f'https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup'
    headers = {
        'User-Agent': 'Mozilla/5.0'
    }

    response = requests.get(web, headers=headers)
    # print(response.status_code)
    content = response.text
    soup = BeautifulSoup(content, 'lxml')

    matches = soup.find_all('div', class_='footballbox')

    home = []
    score = []
    away = []

    for match in matches:
        home.append(match.find('th', class_='fhome').get_text())
        score.append(match.find('th', class_='fscore').get_text())
        away.append(match.find('th', class_='faway').get_text())

    dict_football = {'home': home, 'score': score, 'away': away}
    df_football = pd.DataFrame(dict_football)
    df_football['year'] = year
    return df_football


#historical data
fifa = [get_matches(year) for year in years]
df_fifa = pd.concat(fifa, ignore_index=True)
df_fifa.to_csv('fifa_world_cup_matches.csv', index=False)

#fixture
df_fixture = get_matches(2026)
df_fixture.to_csv('fifa_world_cup_fixture.csv', index=False)

In [1]:
from selenium import webdriver

driver = webdriver.Chrome()

In [2]:
web = 'https://en.wikipedia.org/wiki/1982_FIFA_World_Cup'
driver.get(web)

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
import time
import pandas as pd

driver = webdriver.Chrome()


def get_missing_data(year):
    web = f'https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup'

    driver.get(web)
    matches = driver.find_elements(by='xpath', value='//td[@align="right"]/.. | //td[@style="text-align:right;"]/..')
    # matches = driver.find_elements(by='xpath', value='//tr[@style="font-size:90%"]')

    home = []
    score = []
    away = []

    for match in matches:
        home.append(match.find_element(by='xpath', value='./td[1]').text)
        score.append(match.find_element(by='xpath', value='./td[2]').text)
        away.append(match.find_element(by='xpath', value='./td[3]').text)

    dict_football = {'home': home, 'score': score, 'away': away}
    df_football = pd.DataFrame(dict_football)
    df_football['year'] = year
    time.sleep(2)
    return df_football


years = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974,
         1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014,
         2018, 2022]

test = get_missing_data(2014)
test
len(test)
driver.quit()
#df_fifa = pd.concat(fifa, ignore_index=True)
#df_fifa.to_csv("fifa_worldcup_missing_data.csv", index=False)